In [1]:
import cv2
import numpy as np
import mediapipe as mp
import json
import os
import hashlib

In [2]:
# Step 1: Generate unique template identifier
def get_template_id(image_path):
    """Generate a unique identifier for a template image"""
    with open(image_path, 'rb') as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    return file_hash

# Step 2: Load existing slots from JSON file
def load_slots_from_json(json_file_path):
    """Load existing slot coordinates from JSON file"""
    if not os.path.exists(json_file_path):
        return {}
    
    try:
        with open(json_file_path, 'r') as f:
            return json.load(f)
    except (json.JSONDecodeError, IOError):
        print(f"Warning: Could not load slots from {json_file_path}")
        return {}

# Step 3: Save slots to JSON file
def save_slots_to_json(slots_data, json_file_path):
    """Save slot coordinates to JSON file"""
    try:
        with open(json_file_path, 'w') as f:
            json.dump(slots_data, f, indent=2)
        print(f"Slots saved to {json_file_path}")
    except IOError as e:
        print(f"Error saving slots to {json_file_path}: {e}")

# Step 4: Get or detect slots for template
def get_or_detect_slots(template_path, slots_json_path):
    """
    Get slots for a template - either from existing JSON file or by detecting new ones
    
    Args:
        template_path: Path to the template image
        slots_json_path: Path to the JSON file containing slot coordinates
    
    Returns:
        tuple: (slot_list, output_image, is_newly_detected)
    """
    # Generate unique template ID
    template_id = get_template_id(template_path)
    
    # Load existing slots data
    slots_data = load_slots_from_json(slots_json_path)
    
    # Check if slots already exist for this template
    if template_id in slots_data:
        print(f"Found existing slots for template: {os.path.basename(template_path)}")
        slot_list = slots_data[template_id]
        
        # Load template image for output visualization
        image = cv2.imread(template_path)
        output = image.copy()
        
        # Draw existing slots for visualization
        for slot in slot_list:
            cv2.circle(output, (slot['center_x'], slot['center_y']), slot['radius'], (0, 255, 0), 2)
            cv2.circle(output, (slot['center_x'], slot['center_y']), 2, (0, 0, 255), 3)
        
        return slot_list, output, False
    else:
        print(f"No existing slots found for template: {os.path.basename(template_path)}")
        print("Detecting new slots...")
        
        # Detect new slots
        slot_list, output = detect_circular_slots(template_path)
        
        # Save the new slots
        slots_data[template_id] = slot_list
        save_slots_to_json(slots_data, slots_json_path)
        
        print(f"Detected and saved {len(slot_list)} slots for template")
        return slot_list, output, True

# Step 5: Original slot detection function (unchanged)
def detect_circular_slots(image_path):
    """Detect circular slots in the poster image"""
    image = cv2.imread(image_path)
    output = image.copy()
    
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    
    # Edge detection
    edges = cv2.Canny(blurred, 50, 150)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Store detected slots
    slot_list = []
    
    for contour in contours:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        
        if perimeter == 0:
            continue
            
        circularity = 4 * np.pi * area / (perimeter * perimeter + 1e-6)
        
        # Accept circular shapes with sufficient area
        if circularity > 0.75 and area > 3000:
            (x_center, y_center), radius = cv2.minEnclosingCircle(contour)
            x_center, y_center, radius = int(x_center), int(y_center), int(radius)
            
            # Only accept circles with reasonable radius
            if radius >= 100:
                # Convert to bounding box
                x = x_center - radius
                y = y_center - radius
                width = height = radius * 2
                
                # Add to slot list
                slot = {
                    "x": int(x),
                    "y": int(y),
                    "width": int(width),
                    "height": int(height),
                    "center_x": x_center,
                    "center_y": y_center,
                    "radius": radius,
                    "shape": "circle"
                }
                slot_list.append(slot)
                
                # Draw for visualization
                cv2.circle(output, (x_center, y_center), radius, (0, 255, 0), 2)
                cv2.circle(output, (x_center, y_center), 2, (0, 0, 255), 3)
    
    return slot_list, output

# Step 6: Sort slots by position (unchanged)
def sort_slots_by_position(slots):
    """Sort slots by position for predictable ordering"""
    return sorted(slots, key=lambda s: (s['y'], s['x']))

# Step 7: Improved face cropping function (unchanged)
def crop_face_for_slot(image_path, slot_width, slot_height, white_background=True):
    img = cv2.imread(image_path)
    h, w, _ = img.shape

    # Optional: remove background
    if white_background:
        mp_selfie = mp.solutions.selfie_segmentation
        with mp_selfie.SelfieSegmentation(model_selection=1) as segment:
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = segment.process(rgb_img)
            mask = results.segmentation_mask > 0.5

            white_bg = np.ones_like(img, dtype=np.uint8) * 255
            img = np.where(mask[..., None], img, white_bg)

    # Face detection
    mp_face = mp.solutions.face_detection
    with mp_face.FaceDetection(model_selection=1, min_detection_confidence=0.5) as fd:
        results = fd.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if not results.detections:
            print("No face found in", image_path)
            return None

        bbox = results.detections[0].location_data.relative_bounding_box
        x_center = int((bbox.xmin + bbox.width / 2) * w)
        y_center = int((bbox.ymin + bbox.height / 2) * h)

        x1 = max(0, x_center - slot_width // 2)
        y1 = max(0, y_center - slot_height // 2)
        x2 = min(w, x1 + slot_width)
        y2 = min(h, y1 + slot_height)

        crop = img[y1:y2, x1:x2]
        crop_h, crop_w, _ = crop.shape

        if crop_w < slot_width or crop_h < slot_height:
            scale_w = slot_width / crop_w
            scale_h = slot_height / crop_h
            scale = max(scale_w, scale_h)
            new_w = int(crop_w * scale)
            new_h = int(crop_h * scale)
            crop = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

            start_x = (new_w - slot_width) // 2
            start_y = (new_h - slot_height) // 2
            crop = crop[start_y:start_y + slot_height, start_x:start_x + slot_width]

        return crop

# Step 8: Improved face merging function (unchanged)
def merge_faces(poster_img, face_images, slots):
    """Merge face images into poster slots with better masking"""
    canvas = poster_img.copy()
    
    for i, (face_img, slot) in enumerate(zip(face_images, slots)):
        if face_img is None:
            print(f"Skipping slot {i} - no face image")
            continue
            
        x, y = slot['x'], slot['y']
        w, h = slot['width'], slot['height']
        center_x, center_y = slot['center_x'], slot['center_y']
        radius = slot['radius']
        
        # Ensure face image matches slot dimensions
        if face_img.shape[:2] != (h, w):
            face_img = cv2.resize(face_img, (w, h), interpolation=cv2.INTER_CUBIC)
        
        # Create circular mask
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.circle(mask, (w//2, h//2), radius, 255, -1)
        
        # Apply mask to face image
        face_masked = cv2.bitwise_and(face_img, face_img, mask=mask)
        
        # Create inverse mask for poster
        mask_inv = cv2.bitwise_not(mask)
        poster_region = canvas[y:y+h, x:x+w]
        poster_masked = cv2.bitwise_and(poster_region, poster_region, mask=mask_inv)
        
        # Combine masked face and poster
        combined = cv2.add(face_masked, poster_masked)
        canvas[y:y+h, x:x+w] = combined
        
        print(f"Placed face {i+1} in slot at ({x}, {y})")
    
    return canvas

# Step 9: Main function to process template and face images
def process_template_with_faces(template_path, face_image_paths, slots_json_path, output_path):
    """
    Main function to process a template with face images
    
    Args:
        template_path: Path to the template image
        face_image_paths: List of paths to face images
        slots_json_path: Path to JSON file for storing/loading slot coordinates
        output_path: Path to save the final merged image
    """
    print(f"Processing template: {template_path}")
    
    # Get or detect slots for the template
    slots, template_with_slots, is_newly_detected = get_or_detect_slots(template_path, slots_json_path)
    
    if not slots:
        print("No slots found in template!")
        return None
    
    # Sort slots by position
    sorted_slots = sort_slots_by_position(slots)
    
    print(f"Found {len(sorted_slots)} slots in template")
    
    # Load template image
    template_img = cv2.imread(template_path)
    
    # Process face images
    face_images = []
    for i, face_path in enumerate(face_image_paths):
        if i >= len(sorted_slots):
            print(f"Warning: More face images than slots. Skipping {face_path}")
            break
            
        slot = sorted_slots[i]
        face_crop = crop_face_for_slot(face_path, slot['width'], slot['height'])
        face_images.append(face_crop)
    
    # Merge faces with template
    final_image = merge_faces(template_img, face_images, sorted_slots)
    
    # Save the result
    cv2.imwrite(output_path, final_image)
    print(f"Final merged image saved to: {output_path}")
    
    return final_image

In [3]:
# Example usage:
if __name__ == "__main__":
    # Example paths
    template_path = "2_slot_template.png"
    face_paths = ["p1.jpeg","p2.jpeg"]
    # face_paths = ["p1.jpeg","p2.jpeg","p3.jpeg","p5(with_bg).jpeg"]
    slots_json_path = "template_slots.json"
    output_path = "merged_result.jpg"
    
    # Process the template
    result = process_template_with_faces(
        template_path, 
        face_paths, 
        slots_json_path, 
        output_path
    )

Processing template: 2_slot_template.png
Found existing slots for template: 2_slot_template.png
Found 2 slots in template
Placed face 1 in slot at (165, 1252)
Placed face 2 in slot at (935, 1252)
Final merged image saved to: merged_result.jpg


In [ ]:
#1. Remove details of a slot (if template is deleted from db)